In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [2]:
import torch
from accelerate import Accelerator
from transformers import AutoModelForCausalLM, AutoTokenizer, get_scheduler, DataCollatorWithPadding, TrainingArguments, Trainer
from torch.optim import AdamW
from datasets import load_dataset
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm
import evaluate

## Load Dataset

In [3]:
from datasets import load_dataset

red_pajama = load_dataset("togethercomputer/RedPajama-Data-V2", 'sample', split="train")

Loading dataset shards:   0%|          | 0/22 [00:00<?, ?it/s]

In [4]:
red_pajama

Dataset({
    features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 1050391
})

In [5]:
red_pajama = red_pajama.train_test_split(test_size=0.3)
red_pajama

DatasetDict({
    train: Dataset({
        features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
        num_rows: 735273
    })
    test: Dataset({
        features: ['raw_content', 'doc_id', 'meta', 'quality_signals'],
        num_rows: 315118
    })
})

In [6]:
training_samples = red_pajama['train']
validation_samples = red_pajama['test']
training_samples = training_samples.rename_column('raw_content', 'text')
validation_samples = validation_samples.rename_column('raw_content', 'text')
print(training_samples)
print(validation_samples)

Dataset({
    features: ['text', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 735273
})
Dataset({
    features: ['text', 'doc_id', 'meta', 'quality_signals'],
    num_rows: 315118
})


In [7]:
training_samples[0]

{'text': '15-18 September: Vibrant arts, music, shows and entertainment in and around Manchester\nZak Trojano will perform at the Foundry on September 15th.\nThursday, September 15thth\nZak Trojano / The Foundry (Manchester) / 5pm\nDoug Thompson / T-Bones (Concord) / 5pm\nJosh Foster / Uno Pizzeria & Grill (Concord) / 6pm\nJodee Frawlee / Cheers (Concord) / 6pm\nLouis Goodwin / Elm Street Patio (Manchester) / 6pm\nAmanda Adams / San Francisco Kitchen (Nashua) / 6:30 PM\nKillian Venman Duo / City Hall Pub (Manchester) / 19:00\nMugsy Duo / Stumble Inn (Londonderry) / 19:00\nMobounce will return to Derryfield on 16 September.\nFriday 16th Septemberth\nRuss Six / The Goat Patio (Manchester) / 4pm\nPete Massa / The Hill Bar & Grill (Manchester) / 5:30 pm\nKen Budka / Backyard Brewery (Manchester) / 6pm\nJoannie Cicatelli / Homestead (Merrimack) / 6pm\nJordan Quinn / Firefly (Manchester) / 6pm\nJoe Macdonald / Coach Stop (Londonderry) / 6pm\nMo Bounce / Derryfield (Manchester) / 8pm\nMB Padf

In [8]:
training_samples.column_names

['text', 'doc_id', 'meta', 'quality_signals']

In [9]:
model_name = "facebook/opt-350m"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

In [10]:
# tokenizer.chat_template = (
#     "{% for message in messages %}"
#     "{% if message['role'] == 'system' %}<|system|>\n{{ message['content'] }}\n"
#     "{% elif message['role'] == 'user' %}<|start_header_id|>user<|end_header_id|>{{ message['content'] }}<|eot_id|>"
#     "{% elif message['role'] == 'assistant' %}<|start_header_id|>assistant<|end_header_id|>{{ message['content'] }}<|eot_id|>"
#     "{% endif %}"
#     "{% endfor %}"
# )

## Model Training

In [10]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

# to help save on gpu space and run this a bit faster we'll load the model in 4bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,
    device_map="auto"
)
model.config.pad_token_id = tokenizer.pad_token_id

In [11]:
from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

In [12]:
from peft import LoraConfig
# rank defines the rank of the adapter matrix,
# the higher the rank, the more complex the task it's trying to learn
rank = 128

# the alpha is a scaling factor hyper parameter, basically controls how much our
# adapter will influence the models output, the higher this value
# the more our adapter will overpower the original model weights.
# there is a lot of advice out there for what the alpha value should be
# keeping the alpha at around 2x of what the rank is works for this notebook
alpha = rank*2
peft_config = LoraConfig(
    r=rank,
    lora_alpha=alpha,
    lora_dropout=0.05, # dropout for the lora layers while training, to avoid overfitting
    bias="none",
    task_type="CAUSAL_LM",
    # the target modules defines what types of layers to add lora adapters too, so in the network
    # any model that have a name in this list will have a lora adapter added to it,
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj', 'gate_proj', 'down_proj', 'up_proj']
)

In [13]:
from transformers import TrainingArguments
from trl import SFTTrainer

model_checkpoint_path = "./results/opt-350m"

# an important note is that the loss function isn't defined here,
# it's instead stored as a model parameter for models in hf,
# in the case of llama it is cross entropy loss

# first define some training arguments
training_arguments = TrainingArguments(
    output_dir=model_checkpoint_path,
    optim='adamw_torch', #specify what optimizer we wwant to use, in this case a 8bit version of adamw with pagination.
    per_device_train_batch_size=8, # define the number of samples per training batch
    gradient_accumulation_steps=4, # define how many steps to accumulate gradients,
    log_level='debug',
    eval_strategy = "steps",
    save_strategy='steps', # we'll save a checkpoint every epoch
    logging_steps=8,
    eval_steps=8,
    save_steps=8,
    save_total_limit=2,
    learning_rate=1e-5, # for llm training we want a fairly high learning rate, 1e-4 is a good starting point but it's worth it to play around with this value
    fp16=True,
    num_train_epochs=4,
    max_steps=120,
    warmup_ratio=0.1,
    load_best_model_at_end = True,
    overwrite_output_dir = True,
    lr_scheduler_type='linear',# and set our learning rate decay
)

# now that we have our arguments, we'll use that to create our trainer,
# passing in the model, dataset, peft config, tokenizer, ect
trainer = SFTTrainer(
    model=model,
    train_dataset=training_samples,
    eval_dataset=validation_samples,
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_arguments
)

Converting train dataset to ChatML:   0%|          | 0/735273 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/735273 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/735273 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/735273 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/315118 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/315118 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/315118 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/315118 [00:00<?, ? examples/s]

max_steps is given, it will override any value given in num_train_epochs
Using auto half precision backend
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [15]:
trainer.train(resume_from_checkpoint='./results/opt-350m/checkpoint-32')

Loading model from ./results/opt-350m/checkpoint-32.
There were missing keys in the checkpoint model loaded: ['base_model.model.model.decoder.embed_tokens.weight', 'base_model.model.model.decoder.embed_positions.weight', 'base_model.model.model.decoder.project_out.weight', 'base_model.model.model.decoder.project_in.weight', 'base_model.model.model.decoder.layers.0.self_attn.k_proj.base_layer.weight', 'base_model.model.model.decoder.layers.0.self_attn.k_proj.base_layer.bias', 'base_model.model.model.decoder.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.decoder.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.decoder.layers.0.self_attn.v_proj.base_layer.weight', 'base_model.model.model.decoder.layers.0.self_attn.v_proj.base_layer.bias', 'base_model.model.model.decoder.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.decoder.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.decoder.layers

ValueError: loaded state dict contains a parameter group that doesn't match the size of optimizer's group